# Module 4.1 - Conversation History Management

**Reference:** [`01-conversation-history-management.md`](./01-conversation-history-management.md)

## What you'll do in this notebook

- Watch the token count of an unbounded conversation climb until it hits the context window.
- Implement a **fixed-window** chatbot that keeps the last N messages.
- Implement a **token-window** chatbot that adapts to message length.
- Spot the information each strategy silently throws away.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Unbounded history

Simulate a 30-turn conversation with a bot that never prunes its history. Each turn, print the total number of tokens that would be sent with the next API call.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant. Keep replies to one sentence."

def count_tokens(messages: list[dict]) -> int:
    return sum(len(encoding.encode(m["content"])) for m in messages)

class UnboundedChatbot:
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        self.history: list[dict] = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        resp = client.chat.completions.create(model=MODEL, messages=self.history)
        reply = resp.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        return reply

bot = UnboundedChatbot(SYSTEM_PROMPT)
for i in range(1, 11):  # keep it to 10 to save API calls
    bot.chat(f"Briefly describe fact number {i} about bees.")
    print(f"Turn {i:2}: history has {len(bot.history)-1:2} messages, {count_tokens(bot.history):4} tokens")

## Exercise 2 - Fixed-window chatbot

Keep only the most recent N user/assistant messages (the system prompt is always kept).

In [ ]:
class FixedWindowChatbot:
    def __init__(self, system_prompt: str, window_size: int = 6):
        self.system_prompt = system_prompt
        self.window_size = window_size
        self.history: list[dict] = []

    def _prompt_messages(self, new_user_message: str) -> list[dict]:
        # TODO: return [system] + last window_size messages from self.history + [{user, new_user_message}]
        raise NotImplementedError

    def chat(self, user_message: str) -> str:
        messages = self._prompt_messages(user_message)
        resp = client.chat.completions.create(model=MODEL, messages=messages)
        reply = resp.choices[0].message.content
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": reply})
        return reply

bot = FixedWindowChatbot(SYSTEM_PROMPT, window_size=4)
bot.chat("My account number is ACC-42.")
for i in range(8):
    bot.chat(f"Give me a short fun fact about topic {i}.")

# Now ask about the account number that was stated at turn 1.
print(bot.chat("What is my account number?"))

Did the bot still know the account number? If the window was smaller than the distance back to the original message, it's gone. That's the indiscriminate-forgetting problem.

## Exercise 3 - Token-window chatbot

Prune oldest messages until the total tokens of `history` are under `max_tokens`.

In [ ]:
class TokenWindowChatbot:
    def __init__(self, system_prompt: str, max_tokens: int = 500):
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.history: list[dict] = []

    def _prune(self) -> None:
        # TODO: while count_tokens(self.history) > self.max_tokens and len(self.history) > 2,
        # drop the oldest user/assistant pair (two pops from the front of the list).
        raise NotImplementedError

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        self._prune()
        messages = [{"role": "system", "content": self.system_prompt}] + self.history
        resp = client.chat.completions.create(model=MODEL, messages=messages)
        reply = resp.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        self._prune()
        return reply

bot = TokenWindowChatbot(SYSTEM_PROMPT, max_tokens=300)
for i in range(10):
    bot.chat(f"Short fact about topic {i}.")
    print(f"Turn {i+1:2}: {len(bot.history):2} messages, {count_tokens(bot.history):4} tokens")

## Reflection

- The account-number test from Exercise 2 is the motivating failure for the rest of Module 4.
- Fixed-window and token-window both forget without considering importance.
- Next up: compress old turns into a summary (section 4.2), then score by importance (4.3), then retrieve selectively (4.4).

## What's next

`02-conversation-summarisation.ipynb` replaces blunt pruning with a rolling summary, so the bot remembers the gist of what was said even after the raw messages are gone.